# Solar filament segmentation -- Colab training run `c2` (jobs listed in the first code cell)

One-time setup (about 2 minutes):

1. **Runtime -> Change runtime type -> T4 GPU.**
2. On Kaggle: *Settings -> API -> Generate New Token* and copy it.
   (Do **not** use "Create Legacy API Key": that would expire the key already in use.)
3. In Colab, open the key icon (Secrets) in the left sidebar and add **one** secret with
   *Notebook access* switched on: `KAGGLE_API_TOKEN` = the token from step 2.
   (If Secrets are unavailable, e.g. in the VS Code Colab extension, the notebook asks
   for the token in a hidden prompt instead. The token is never printed or saved.)
4. **Runtime -> Run all.**

If the session disconnects, reconnect and *Run all* again: training resumes from the
last checkpoint, which is stored in the private Kaggle dataset `fil-colab-c2`.
Results (weights, logs, validation scores, submission) are uploaded to the same
private dataset automatically; nothing needs to be downloaded by hand.


In [ ]:
RUN = 'c2'
JOBS = [('r34-2048-f0', ['--fold', '0', '--res', '2048', '--crop', '768', '--epoch-mult', '4', '--encoder', 'resnet34', '--norm', 'disk', '--epochs', '30', '--bs', '8', '--infer-bs', '2', '--lr', '3e-4', '--wd', '1e-4', '--ema', '0.999', '--val-every', '10', '--tta', '4', '--workers', '2', '--max-hours', '1.7', '--predict-test']), ('v2s-2048-f0', ['--fold', '0', '--res', '2048', '--crop', '768', '--epoch-mult', '4', '--encoder', 'tu-tf_efficientnetv2_s', '--norm', 'disk', '--epochs', '30', '--bs', '6', '--infer-bs', '2', '--lr', '3e-4', '--wd', '1e-4', '--ema', '0.999', '--val-every', '10', '--tta', '4', '--workers', '2', '--max-hours', '2.0', '--predict-test'])]  # (name, filseg.train arguments), run one after another
SYNC_MINUTES = 20  # upload the checkpoint at most this often
DEFAULT_KAGGLE_USER = 'vivianhsu940827'  # public Kaggle user name; override with the optional secret KAGGLE_USERNAME


In [ ]:
import os, subprocess, sys, json, time, shutil, threading, base64, io, zipfile, getpass
from pathlib import Path
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle", "segmentation-models-pytorch", "pycocotools"], check=True)

def _secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:  # secret missing, notebook access off, or userdata unsupported
        return None

TOKEN = _secret("KAGGLE_API_TOKEN") or getpass.getpass("KAGGLE_API_TOKEN (hidden input): ").strip()
if not TOKEN:
    raise RuntimeError("KAGGLE_API_TOKEN is empty: add the Colab secret KAGGLE_API_TOKEN or paste the token at the prompt.")
USER = _secret("KAGGLE_USERNAME") or DEFAULT_KAGGLE_USER
SLUG = f"{USER}/fil-colab-{RUN}"
WORK = Path("/content/work"); OUT = WORK / "out" / RUN; OUT.mkdir(parents=True, exist_ok=True)  # one sub-folder per job

def kaggle(*a, check=True):
    # the token is passed only to the kaggle CLI subprocess, never to os.environ or the training process
    env = dict(os.environ, KAGGLE_API_TOKEN=TOKEN, KAGGLE_USERNAME=USER)
    r = subprocess.run(["kaggle", *a], capture_output=True, text=True, env=env)
    r.stdout, r.stderr = r.stdout.replace(TOKEN, "***"), r.stderr.replace(TOKEN, "***")
    if check and r.returncode:
        raise RuntimeError(r.stdout + r.stderr)
    return r

_auth = kaggle("competitions", "files", "filament-segmentation-2026", "--page-size", "1", check=False)
if _auth.returncode != 0:
    raise RuntimeError("Kaggle authentication failed. Check the Colab secret KAGGLE_API_TOKEN "
                     "(Secrets panel, Notebook access ON) or paste a valid token at the prompt.\n"
                     + (_auth.stderr or _auth.stdout)[-300:])
print("Kaggle authentication OK for", USER)


In [ ]:
DATA = WORK / "data"
if not (DATA / "MAGFiLO_1.0_Kaggle_2026").exists():
    kaggle("competitions", "download", "filament-segmentation-2026", "-p", str(DATA), "-q")
    for z in DATA.glob("*.zip"):
        zipfile.ZipFile(z).extractall(DATA); z.unlink()
print(sorted(p.name for p in (DATA / "MAGFiLO_1.0_Kaggle_2026").iterdir()))


In [ ]:
PAYLOAD = "..."
SRC = WORK / "src"; shutil.rmtree(SRC, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(SRC)
print("code unpacked to", SRC)


In [ ]:
SYNC = WORK / "sync"
FINAL_FILES = ["model.pt", "post.json", "log.txt", "val_probs.npz", "test_probs.npz", "submission.csv",
               "timing.json", "done.json"]

def dataset_exists():
    return kaggle("datasets", "status", SLUG, check=False).returncode == 0

# after a disconnect: restore finished jobs and the checkpoint of the unfinished one
if dataset_exists() and not any((OUT / name).exists() for name, _ in JOBS):
    tmp = WORK / "resume"; shutil.rmtree(tmp, ignore_errors=True)
    kaggle("datasets", "download", SLUG, "-p", str(tmp), "--unzip", check=False)
    for name, _ in JOBS:
        for f in tmp.glob(f"{name}__*"):
            (OUT / name).mkdir(parents=True, exist_ok=True)
            shutil.copy(f, OUT / name / f.name.split("__", 1)[1])
for name, _ in JOBS:
    d = OUT / name
    state = "done" if (d / "done.json").exists() else ("resume" if (d / "last.pt").exists() else "new")
    print(f"{name}: {state}")


In [ ]:
_lock = threading.Lock()
def sync(message, with_checkpoint):
    with _lock:
        shutil.rmtree(SYNC, ignore_errors=True); SYNC.mkdir(parents=True)
        for name, _ in JOBS:
            d = OUT / name
            done = (d / "done.json").exists()
            for f in FINAL_FILES + (["last.pt"] if with_checkpoint and not done else []):
                if (d / f).exists():
                    shutil.copy(d / f, SYNC / f"{name}__{f}")
        (SYNC / "dataset-metadata.json").write_text(json.dumps(
            {"title": f"fil-colab-{RUN}", "id": SLUG, "licenses": [{"name": "other"}]}))
        if dataset_exists():
            r = kaggle("datasets", "version", "-p", str(SYNC), "-m", message, "-q", check=False)
        else:
            r = kaggle("datasets", "create", "-p", str(SYNC), "-q", check=False)  # private by default
        print(f"[sync] {message}: rc={r.returncode} {r.stderr.strip()[-200:]}", flush=True)

stop = threading.Event()
def periodic():
    last = None
    while not stop.wait(SYNC_MINUTES * 60):
        stamps = tuple((OUT / n / "last.pt").stat().st_mtime for n, _ in JOBS if (OUT / n / "last.pt").exists())
        if stamps and stamps != last:
            last = stamps
            sync(f"checkpoint {time.strftime('%H:%M')}", with_checkpoint=True)
threading.Thread(target=periodic, daemon=True).start()

env = dict(os.environ, PYTHONPATH=str(SRC), PYTHONUNBUFFERED="1")
codes = {}
for name, args in JOBS:
    d = OUT / name
    if (d / "done.json").exists():
        print(f"===== {name}: already finished, skipping"); continue
    print(f"===== {name}: {' '.join(args)}", flush=True)
    cmd = [sys.executable, "-m", "filseg.train", "--data", str(DATA / "MAGFiLO_1.0_Kaggle_2026"), "--out", str(d), *args]
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    codes[name] = proc.wait()
    print(f"===== {name}: exit code {codes[name]}", flush=True)
    sync(f"{name} finished rc={codes[name]}", with_checkpoint=False)
stop.set()
print("all jobs finished:", codes)
